In [277]:
import numpy as np
import pandas as pd
import cvxopt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split , KFold
from sklearn.metrics import accuracy_score

from cvxopt import matrix, solvers
from sklearn.base import BaseEstimator, ClassifierMixin

In [278]:
import pandas as pd
data = pd.read_csv("data_2nd_map.csv")

In [279]:
# data['Binding Energy '] = abs(data['Binding Energy '])
import matplotlib.pyplot as plt

# Imports from Qiskit
from qiskit import transpile , transpiler
from qiskit_aer import AerSimulator 
from qiskit import QuantumCircuit
from qiskit.circuit.library import GroverOperator, MCMT, ZGate
from qiskit.visualization import plot_distribution

# Imports from Qiskit Runtime
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [280]:
import numpy as np
import pandas as pd
import cvxopt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split , KFold
from sklearn.metrics import accuracy_score

In [281]:
from sklearn.model_selection import train_test_split
X = data.drop(columns=['Otsu Class theshold -77.8'])
y = data['Otsu Class theshold -77.8']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.5, shuffle=True, random_state=19
)


folds_data = []
n_folds = 4
for fold in range(n_folds):
    
    X_tr, X_v, y_tr, y_v = train_test_split(
        X, y, test_size=0.5, shuffle=True, random_state=19 + fold
    )
    folds_data.append({
        'fold': fold + 1,
        'X_train': X_tr,
        'y_train': y_tr,
        'X_val': X_v,
        'y_val': y_v
    })


In [282]:

# Fold 1
fold_1_X_train = folds_data[0]['X_train']
fold_1_y_train = folds_data[0]['y_train']
fold_1_X_val = folds_data[0]['X_val']
fold_1_y_val = folds_data[0]['y_val']
# fold 2
fold_2_X_train = folds_data[1]['X_train']
fold_2_y_train = folds_data[1]['y_train']
fold_2_X_val = folds_data[1]['X_val']
fold_2_y_val = folds_data[1]['y_val']
# fold 3
fold_3_X_train = folds_data[2]['X_train']
fold_3_y_train = folds_data[2]['y_train']
fold_3_X_val = folds_data[2]['X_val']
fold_3_y_val = folds_data[2]['y_val']
# fold 4
fold_4_X_train = folds_data[3]['X_train']
fold_4_y_train = folds_data[3]['y_train']
fold_4_X_val = folds_data[3]['X_val']
fold_4_y_val = folds_data[3]['y_val']

In [283]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))

#standarize data for training
scaler.fit(fold_1_X_train)
scaler.fit(fold_2_X_train)
scaler.fit(fold_3_X_train)
scaler.fit(fold_4_X_train)

# standarize data for validation
scaler.fit(fold_1_X_val)
scaler.fit(fold_2_X_val)
scaler.fit(fold_3_X_val)
scaler.fit(fold_4_X_val)

MinMaxScaler(feature_range=(-3.141592653589793, 3.141592653589793))

In [284]:
# standarize variable for training
standar_fold_1_X_train = scaler.transform(fold_1_X_train)
standar_fold_2_X_train = scaler.transform(fold_2_X_train)
standar_fold_3_X_train = scaler.transform(fold_3_X_train)
standar_fold_4_X_train = scaler.transform(fold_4_X_train)
# standarize variable for vaidation
standar_fold_1_X_val = scaler.transform(fold_1_X_val)
standar_fold_2_X_val = scaler.transform(fold_2_X_val)
standar_fold_3_X_val = scaler.transform(fold_3_X_val)
standar_fold_4_X_val = scaler.transform(fold_4_X_val)

In [285]:
print(len(fold_1_X_val))

40


In [286]:
data.head()

,Binding Energy,Otsu Class theshold -77.8,Abs dist to threshold,a_1,a_2,a_3,a_4,a_5,a_6,a_7,a_8,a_9
0,-92.5,1,14.7,-4.5,4.2,-1.6,4.5,3.8,-4.5,-0.7,4.2,2.8
1,-77.1,0,0.7,4.2,-1.6,4.5,3.8,-4.5,-0.7,4.2,2.8,-0.4
2,-99.6,1,21.8,-4.5,-0.7,4.2,2.8,-0.4,2.8,-4.5,-0.9,3.8
3,-48.5,0,29.3,-1.6,-0.7,-1.6,-1.6,-3.5,-4.5,-3.5,1.8,3.8
4,-50.1,0,27.7,3.8,3.8,-0.8,-0.8,-3.9,-0.7,-0.8,4.2,1.8


In [287]:
train_size = 40
X_train = standar_fold_1_X_train 
train_labels = fold_1_y_train

In [288]:
# Prepare testing data
test_size = 40
X_test =  standar_fold_1_X_val
test_labels = fold_1_y_val

In [289]:
# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)
test_matrix = np.full((test_size, num_samples), np.nan)

In [ ]:
from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.circuit.library import z_feature_map , ZZFeatureMap , PauliFeatureMap
# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = 5
 
# To use a custom feature map use the lines below.
entangler_map = [[0, 2], [3, 4], [1, 4], [2, 3]]
 
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

In [ ]:
print(np.shape(X_train)[1]/2)
# aqui deberia ser 5

5.5


In [292]:
print(len(list(X_train[x1])))
print(train_size)

11
40


In [293]:
# To use a simulator
from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import unitary_overlap

# service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=fm.num_qubits
# )
 
num_shots = 10000
 
# Evaluate the problem using state vector-based primitives from Qiskit.
sampler = StatevectorSampler()
 
for x1 in range(0, train_size):
    for x2 in range(x1 + 1, train_size):
        unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi/2 ])
        unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi /2])
 
        # Create the overlap circuit
        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()
 
        # These lines run the qiskit sampler primitive.
        counts = (
            sampler.run([overlap_circ], shots=num_shots)
            .result()[0]
            .data.meas.get_int_counts()
        )
 
        # Assign the probability of the 0 state to the kernel matrix, and the transposed element (since this is an inner product)
        kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
        kernel_matrix[x2, x1] = counts.get(0, 0.0) / num_shots
    # Fill in on-diagonal elements with 1, again, since this is an inner-product corresponding to probability (or alter the code to check these entries and verify they yield 1)
    kernel_matrix[x1, x1] = 1
 
print("training done")
 
# Similar process to above, but for testing data.
for x1 in range(0, test_size):
    for x2 in range(0, train_size):
        unitary1 = fm.assign_parameters(list(X_test[x1]) + [np.pi/2 ])
        unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi /2])
 
        # Create the overlap circuit
        overlap_circ = unitary_overlap(unitary1, unitary2)
        overlap_circ.measure_all()
 
        counts = (
            sampler.run([overlap_circ], shots=num_shots)
            .result()[0]
            .data.meas.get_int_counts()
        )
 
        test_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
 
print("test matrix done")

ValueError: Mismatching number of values and parameters. For partial binding please pass a mapping of {parameter: value} pairs.

In [ ]:
# import a support vector classifier from a classical ML package.
from sklearn.svm import SVC
qml_svc = SVC(kernel="precomputed")

In [ ]:
# Feed in the pre-computed matrix and the labels of the training data. The classical algorithm gives you a fit.
qml_svc.fit(kernel_matrix, train_labels)
 
# Now use the .score to test your data, using the matrix of test data, and test labels as your inputs.
qml_score_precomputed_kernel = qml_svc.score(test_matrix, test_labels)
print(f"Precomputed kernel classification test score: {qml_score_precomputed_kernel}")

Precomputed kernel classification test score: 0.625
